# LLM Function Call 示例

本 Notebook 演示 **大模型工具调用（Tool / Function Calling）** 的完整流程：用户提问 → 模型决定调用哪个函数 → 本地执行函数 → 模型根据结果生成最终回答。

## 核心组件

| 组件 | 说明 |
|------|------|
| `tools` | 向模型注册的函数 Schema（名称、描述、参数） |
| `get_weather()` | 本地实际执行的 Mock 天气查询函数 |
| `send_messages()` | 封装 API 调用，传入 `messages_history` + `tools` |
| `messages_history` | 多轮对话历史，贯穿整个调用链 |

## 代码逻辑图

```mermaid
flowchart LR
    A[用户提问] --> B[LLM<br/>返回 tool_call]
    B --> C[执行<br/>get_weather]
    C --> D[LLM<br/>生成回答]
```


In [1]:
from openai import OpenAI
import json
from pprint import pprint
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

def to_api_message(msg):
    """统一转为 API 可接受的 dict 格式。"""
    if hasattr(msg, "model_dump"):
        return msg.model_dump(exclude_none=True)
    return msg

def send_messages(messages_history, tools):
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[to_api_message(m) for m in messages_history],
        tools=tools
    )
    return response.choices[0].message

def print_message(message, style="json"):
    """格式化打印单条消息（dict 或 ChatCompletionMessage），自动去掉 None 字段。"""
    if hasattr(message, "model_dump"):
        data = message.model_dump(exclude_none=True)
    else:
        data = {k: v for k, v in message.items() if v is not None}
    if style == "pprint":
        pprint(data)
    else:
        print(json.dumps(data, indent=2, ensure_ascii=False))

def print_messages(messages_history, style="json"):
    for i, msg in enumerate(messages_history):
        role = msg.get("role") if isinstance(msg, dict) else getattr(msg, "role", "assistant")
        print(f"--- [{i}] {role} ---")
        print_message(msg, style)


## hasattr

`hasattr(msg, "model_dump")`它检测的是 **msg 这个对象上有没有一个叫 `model_dump` 的属性/方法**。

### hasattr 的机制

`hasattr(obj, "name")` 的语义是：**试着访问 `obj.name`，能访问到返回 `True`，抛 `AttributeError` 返回 `False`**。不只是查一层属性，而是整个属性查找链——对象自身的属性、类属性、父类的方法，都会被查到。

```python
hasattr("abc", "upper")        # True，字符串有 upper 方法
hasattr(message, "model_dump") # True，OpenAI 的 message 是 Pydantic 模型
hasattr({"role": "user"}, "model_dump")  # False，普通 dict 没有
```

### 这段代码在干什么

这是一个典型的 **鸭子类型判断**：不检查 `msg` 的具体类型（不用 `isinstance(msg, ChatCompletionMessage)`），只看它「像不像」一个 Pydantic 模型——会 `model_dump` 就当它是。

```python
def to_api_message(msg):
    """统一转为 API 可接受的 dict 格式。"""
    if hasattr(msg, "model_dump"):   # 是 Pydantic 模型对象吗？
        return msg.model_dump(exclude_none=True)  # 是 → 转成干净的 dict
    return msg                        # 不是（本身就是 dict）→ 原样返回
```

它解决的实际问题是：你的 `messages` 列表里可能混着两种东西——

1. **你自己写的 dict**：`{"role": "user", "content": "hi"}`
2. **上一轮 API 返回的 message 对象**（多轮对话时把 assistant 的回复塞回历史）

API 只吃 dict，不吃对象。所以这个函数像一个「适配器」：是对象就转 dict，本来就是 dict 就直接过。

### 为什么用 hasattr 而不是 isinstance

`isinstance` 要求你导入并绑定具体的类（`openai.types.chat.ChatCompletionMessage`），一旦 SDK 升级改了类路径，代码就碎。`hasattr` 只依赖「有没有这个方法」这个行为特征，更宽松、更抗变化——这就是「看起来像鸭子、叫起来像鸭子，那就当它是鸭子」。

代价是理论上可能误判：任何恰好定义了 `model_dump` 方法的其他对象也会走第一条分支。但 `model_dump` 这个名字基本是 Pydantic 的招牌，实际碰撞概率可以忽略。

In [2]:
def get_weather(location: str) -> str:
    """Mock weather lookup — replace with a real API call if needed."""
    mock_data = {
        "hangzhou": "24℃，晴，东南风 3 级",
        "beijing": "18℃，多云，北风 2 级",
        "shanghai": "26℃，阴，东风 2 级",
    }
    key = location.lower().split(",")[0].strip()
    return mock_data.get(key, f"{location} 的天气暂无数据")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
]    

In [3]:
messages_history = [
    {
        "role": "user",
        "content": "How's the weather in Hangzhou, Zhejiang?"
    }
]
message = send_messages(messages_history, tools)
print(f"User>\t {messages_history[0]['content']}")

User>	 How's the weather in Hangzhou, Zhejiang?


In [4]:
print_message(message)

{
  "content": "I'll check the weather in Hangzhou for you.",
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_00_EFsQE0dFDcTKN2dFlLqe7545",
      "function": {
        "arguments": "{\"location\": \"Hangzhou, Zhejiang\"}",
        "name": "get_weather"
      },
      "type": "function",
      "index": 0
    }
  ],
  "reasoning_content": "The user wants the weather in Hangzhou, Zhejiang. Let me call the weather tool."
}


<img src="function-calling-sequence.svg" alt="Function Calling 消息时序图" width="680" style="max-width:100%">


`send_messages(messages_history, tools)` 里的 `tools` 只是**函数说明书**——告诉模型「我这有 get_weather 可用，参数长这样」，相当于递给服务员一份菜单。

模型收到后**并不会自己执行**函数。它返回的 `message` 里 `content` 是空的，`tool_calls` 里装的是一张**点单**：「我想调 get_weather，参数是 `{"location": "Hangzhou, Zhejiang"}`」——菜还没做。真正执行 `get_weather()` 的是你本地的代码（cell-7 里的 `json.loads` + `get_weather(...)`）。

两个关键点：

**1. API 是无状态的。** 第 1 次调用结束后，服务器不记得刚才发生过什么。cell-10 发起第 2 次调用时，如果只发 `[user]`，模型面对的就像一个全新对话——它不知道自己点过单，更不知道天气结果。所以必须把完整的来龙去脉重新发一遍。

**2. 协议要求成对出现。** OpenAI 风格的 API 硬性规定：`role: "tool"` 的结果消息，前面**必须**紧跟一条带 `tool_calls` 的 assistant 消息，且 `tool_call_id` 要对得上，缺了直接报 400。所以顺序是：

```python
messages_history.append(message.model_dump(exclude_none=True))  # 先：assistant 的点单
# ...本地执行函数...
messages_history.append({"role": "tool", "tool_call_id": tool.id, ...})  # 后：上菜结果
```

跑完 cell-9 打印历史，应该是三段式：

```
[0] user      → "How's the weather in Hangzhou?"
[1] assistant → tool_calls: get_weather({"location": ...})   ← 你问的那行
[2] tool      → "24℃，晴，东南风 3 级"
```

第 2 次调用把这三条一起发回去，模型才能「看到自己点过单、看到了结果」，然后生成最终的自然语言回答。

**一句话总结**：`tools` 是每次调用都要带的说明书，`message` 是模型的一次点单；追加 assistant 消息不是重复发送，而是**把点单记入账本**——第 2 次调用时模型靠这条记录才知道自己要调的是什么函数、结果对应哪次请求。


In [5]:
tool = message.tool_calls[0]
# 必须先追加 assistant（含 tool_calls），再追加 tool 结果
messages_history.append(message.model_dump(exclude_none=True))

messages_history

[{'role': 'user', 'content': "How's the weather in Hangzhou, Zhejiang?"},
 {'content': "I'll check the weather in Hangzhou for you.",
  'role': 'assistant',
  'tool_calls': [{'id': 'call_00_EFsQE0dFDcTKN2dFlLqe7545',
    'function': {'arguments': '{"location": "Hangzhou, Zhejiang"}',
     'name': 'get_weather'},
    'type': 'function',
    'index': 0}],
  'reasoning_content': 'The user wants the weather in Hangzhou, Zhejiang. Let me call the weather tool.'}]

In [6]:
# 解析 LLM 返回的函数调用参数，执行真正的 get_weather
tool_args = json.loads(tool.function.arguments)
weather_result = get_weather(tool_args["location"])
messages_history.append({"role": "tool", "tool_call_id": tool.id, "content": weather_result})

In [7]:
print(weather_result)

24℃，晴，东南风 3 级


In [8]:
print_messages(messages_history)

--- [0] user ---
{
  "role": "user",
  "content": "How's the weather in Hangzhou, Zhejiang?"
}
--- [1] assistant ---
{
  "content": "I'll check the weather in Hangzhou for you.",
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_00_EFsQE0dFDcTKN2dFlLqe7545",
      "function": {
        "arguments": "{\"location\": \"Hangzhou, Zhejiang\"}",
        "name": "get_weather"
      },
      "type": "function",
      "index": 0
    }
  ],
  "reasoning_content": "The user wants the weather in Hangzhou, Zhejiang. Let me call the weather tool."
}
--- [2] tool ---
{
  "role": "tool",
  "tool_call_id": "call_00_EFsQE0dFDcTKN2dFlLqe7545",
  "content": "24℃，晴，东南风 3 级"
}


In [9]:
message = send_messages(messages_history, tools)
print(f"Model>\t {message.content}")

Model>	 Here's the current weather in **Hangzhou, Zhejiang** (杭州，浙江):

- 🌡️ **Temperature:** 24°C
- ☀️ **Conditions:** Clear (晴)
- 🌬️ **Wind:** Southeast wind, level 3 (东南风 3 级)

It's a pleasant, clear day — great weather to be outdoors! Let me know if you'd like the forecast for any other location.


In [10]:
print_message(message)

{
  "content": "Here's the current weather in **Hangzhou, Zhejiang** (杭州，浙江):\n\n- 🌡️ **Temperature:** 24°C\n- ☀️ **Conditions:** Clear (晴)\n- 🌬️ **Wind:** Southeast wind, level 3 (东南风 3 级)\n\nIt's a pleasant, clear day — great weather to be outdoors! Let me know if you'd like the forecast for any other location.",
  "role": "assistant",
  "reasoning_content": "The weather result came back in Chinese: 24°C, clear, southeast wind level 3. Let me present it."
}
